In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [35]:
block_size = 8
n_embed = 32

In [32]:
q_pos = torch.arange(block_size, dtype=torch.long)
q_pos = q_pos.view(-1, 1)
k_pos = torch.arange(block_size, dtype=torch.long)
rel_pos = k_pos - q_pos
-rel_pos
# we always use previous tokens , not the future tokens

tensor([[ 0, -1, -2, -3, -4, -5, -6, -7],
        [ 1,  0, -1, -2, -3, -4, -5, -6],
        [ 2,  1,  0, -1, -2, -3, -4, -5],
        [ 3,  2,  1,  0, -1, -2, -3, -4],
        [ 4,  3,  2,  1,  0, -1, -2, -3],
        [ 5,  4,  3,  2,  1,  0, -1, -2],
        [ 6,  5,  4,  3,  2,  1,  0, -1],
        [ 7,  6,  5,  4,  3,  2,  1,  0]])

In [33]:
rel_pos = -rel_pos
tril = torch.tril(torch.ones(block_size, block_size))
rel_pos = rel_pos.masked_fill(tril == 0, 0)
rel_pos


tensor([[0, 0, 0, 0, 0, 0, 0, 0],
        [1, 0, 0, 0, 0, 0, 0, 0],
        [2, 1, 0, 0, 0, 0, 0, 0],
        [3, 2, 1, 0, 0, 0, 0, 0],
        [4, 3, 2, 1, 0, 0, 0, 0],
        [5, 4, 3, 2, 1, 0, 0, 0],
        [6, 5, 4, 3, 2, 1, 0, 0],
        [7, 6, 5, 4, 3, 2, 1, 0]])

In [37]:
embedding = nn.Embedding(block_size, n_embed)
rel_emb = embedding(rel_pos)
rel_emb.shape

torch.Size([8, 8, 32])

In [39]:
#2 * T - 1
pos_emb = torch.randn(2 * block_size - 1, n_embed)
pos_emb.shape

torch.Size([15, 32])

In [72]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class RelativePositionalEncoding(nn.Module):
    def __init__(self, max_rel_dist, dim):
        super().__init__()
        self.max_rel_dist = max_rel_dist
        self.embedding = nn.Parameter(
            torch.randn(2 * max_rel_dist + 1, dim)
        )
    
    def forward(self, length):
        pos_i = torch.arange(length)[:, None]
        pos_j = torch.arange(length)[None, :]
        rel_pos = pos_j - pos_i
        rel_pos = rel_pos.clamp(-self.max_rel_dist, self.max_rel_dist)
        rel_pos = rel_pos + self.max_rel_dist
        return self.embedding[rel_pos]  # [T, T, dim]

class RelativeSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, max_rel_dist):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv_proj = nn.Linear(embed_dim, 3 * embed_dim)
        self.rpe = RelativePositionalEncoding(max_rel_dist, self.head_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
    
    def forward(self, x):
        B, T, _ = x.size()
        qkv = self.qkv_proj(x)  # [B, T, 3*D]
        q, k, v = qkv.split(self.embed_dim, dim=-1)
        #q, k, v = qkv.chunk(3, dim=-1)
        
        # Reshape and transpose for attention heads
        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Scaled dot-product attention scores: [B, H, T, T]
        attn_scores = torch.matmul(q, k.transpose(-1, -2)) * self.scale
        
        # Generate relative positional encodings and sum to attention
        # rel_encoding: [T, T, head_dim]
        rel_encoding = self.rpe(T)
        rel_bias = rel_encoding.permute(2, 0, 1)  # [head_dim, T, T]
        # Project query to last dimension, sum over head_dim
        rel_logits = torch.einsum('bhld,dlm->bhlm', q, rel_bias)  # [B, H, T, T]
        
        attn_scores = attn_scores + rel_logits
        
        attn_weights = F.softmax(attn_scores, dim=-1)
        out = torch.matmul(attn_weights, v)
        out = out.transpose(1, 2).reshape(B, T, self.embed_dim)
        out = self.out_proj(out)
        return out

# Example usage:
C = 32
num_heads = 4
max_rel_dist = 4
B = 4
T = 8

# Input [batch, T, embed_dim]
x = torch.randn(B, T, C)
self_attn = RelativeSelfAttention(C, num_heads, max_rel_dist)
output = self_attn(x)  # [batch, T, embed_dim]
print(output.shape)


torch.Size([4, 8, 32])


In [114]:
import torch
import torch.nn as nn

class RelativePositionalEmbedding(nn.Module):
    def __init__(self, max_relative_distance, head_size):
        super().__init__()
        # max_relative_distance should be twice the max sequence length - 1
        # to cover all possible relative distances (positive and negative)
        self.embeddings = nn.Embedding(max_relative_distance * 2 - 1, head_size)

    def forward(self, relative_indices):
        return self.embeddings(relative_indices)

In [117]:
def generate_relative_indices(seq_len):
    range_tensor = torch.arange(seq_len)
    # Create a matrix of relative distances
    # Example: for seq_len=3, this would be:
    # [[ 0,  1,  2],
    #  [-1,  0,  1],
    #  [-2, -1,  0]]
    relative_indices = range_tensor.unsqueeze(0) - range_tensor.unsqueeze(1)
    # Shift indices to be non-negative for embedding lookup
    # Add max_relative_distance - 1 to shift negative values
    max_relative_distance = seq_len # For simplicity, assuming max_relative_distance = seq_len
    relative_indices += (max_relative_distance - 1)
    return relative_indices

In [124]:
indices = generate_relative_indices(8)
rpe = RelativePositionalEmbedding(8, 4)
rpe(indices).shape

torch.Size([8, 8, 4])

In [ ]:
class SelfAttentionWithRelativePE(nn.Module):
    def __init__(self, embed_dim, num_heads, max_relative_distance):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.qkv_proj = nn.Linear(embed_dim, 3 * embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)

        self.relative_pe = RelativePositionalEmbedding(max_relative_distance, self.head_dim)

    def forward(self, x):
        batch_size, seq_len, _ = x.shape

        qkv = self.qkv_proj(x)  # [B, T, 3*D]
        q, k, v = qkv.split(self.embed_dim, dim=-1)

        # Project and reshape for multi-head attention
        Q = q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = k.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = v.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        print("Q", Q.shape)

        # Calculate attention scores
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5) # (B, nh, T, hs) @ (B, nh, hs, T) => (B, nh, T, T)

        # Add relative positional bias
        relative_indices = generate_relative_indices(seq_len)
        # Expand relative_indices for broadcasting across heads and batch
        relative_bias = self.relative_pe(relative_indices).permute(2, 0, 1).unsqueeze(0) # (1, H, S, S)
        print("relative_bias", relative_bias.shape)
        print("scores", scores.shape)
        #scores = scores + relative_bias

        #if mask is not None:
        #    scores = scores.masked_fill(mask[:, :, T, T] == 0, float('-inf'))

        attention_weights = torch.softmax(scores, dim=-1)
        output = torch.matmul(attention_weights, V)

        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.embed_dim)
        return self.out_proj(output)

In [131]:
self_att = SelfAttentionWithRelativePE(embed_dim=n_embed, num_heads=8, max_relative_distance=block_size)
mask = torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size)
out = self_att(x)
out.shape

Q torch.Size([4, 8, 8, 4])
relative_bias torch.Size([1, 4, 8, 8])
scores torch.Size([4, 8, 8, 8])


torch.Size([4, 8, 32])